# Human Protein Sequence Retrieval

>Retrieves reviewed amino acid sequences from UniProt for all human proteins in the PPI dataset 
> and exports a sequence-annotated CSV.

In [7]:
import pandas as pd
import requests
from tqdm import tqdm

## Configuration and Paths

In [15]:
INPUT_FILE  = "/Users/matteo/Desktop/Uttopia_all_shit/ALL_CODE_/A_Dataset_Human/0_All_protein_human.csv"
OUTPUT_FILE = "/Users/matteo/Desktop/Uttopia_all_shit/ALL_CODE_/A_Dataset_Human/1_All_protein_uniprot_human.csv"

PROT_COL = "gene_name"
TAXON_ID = "9606"  # Homo sapiens

## UniProt Proteome Retrieval

In [16]:
def get_human_database():
    """
    Downloads the reviewed Homo sapiens proteome from UniProt.
    Returns a dictionary mapping Gene symbols/aliases to amino acid sequences.
    """
    
    url = "https://rest.uniprot.org/uniprotkb/stream"
    params = {
        'format': 'tsv',
        'fields': 'gene_names,sequence',
        'query': f'taxonomy_id:{TAXON_ID} AND reviewed:true'
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    local_map = {}
    lines = response.text.strip().split('\n')
    
    for line in lines[1:]:
        parts = line.split('\t')
        if len(parts) >= 2:
            aliases = parts[0].upper().split() 
            sequence = parts[1]
            for alias in aliases:
                local_map[alias] = sequence
                
    return local_map

## Load & Pre-process 

In [17]:
human_db = get_human_database()
df = pd.read_csv(INPUT_FILE)

In [18]:
df.shape

(22789, 1)

## Sequence Mapping

In [20]:
unique_genes = pd.concat([df[PROT_COL]]).unique()
gene_to_seq_map = {}

for gene in tqdm(unique_genes, desc="Mapping sequences"):
    if pd.isna(gene):
        gene_to_seq_map[gene] = "NOT_FOUND"
    else:
        name = str(gene).strip().upper()
        gene_to_seq_map[gene] = human_db.get(name, "NOT_FOUND")

df['seq_prot'] = df[PROT_COL].map(gene_to_seq_map)

Mapping sequences: 100%|██████████| 22788/22788 [00:00<00:00, 840195.50it/s]


## Quality Check & Export

In [21]:
n_missing = (df["seq_prot"] == "NOT_FOUND").sum()
print(f"Sequences not found: {n_missing} / {len(df)}")


df_clean = df[df["seq_prot"] != "NOT_FOUND"].copy()
print(f"Proteins retained:   {len(df_clean)}")

df_clean.to_csv(OUTPUT_FILE, index=False)
print(f"Saved → {OUTPUT_FILE} :)")

Sequences not found: 1538 / 22789
Proteins retained:   21251
Saved → /Users/matteo/Desktop/Uttopia_all_shit/ALL_CODE_/A_Dataset_Human/1_All_protein_uniprot_human.csv :)
